# 01 — Análise Exploratória dos Dados (EDA)

**Área de análise:** diagnóstico de hipotireoidismo  
**Base de dados:** `Hypothyroid.csv`  
**Variável-alvo:** `binaryClass` (`P` e `N`)  
**Etapa seguinte:** preparação dos dados e modelagem no notebook **[02_ML](02_ML.ipynb)**


**OBJETIVO**

Este notebook realiza a análise exploratória do conjunto `Hypothyroid.csv`, com foco na compreensão da estrutura dos dados, avaliação da qualidade das variáveis e investigação de diferenças entre as classes `P` e `N`. Ao final, são documentadas as decisões de preparação e gerados os conjuntos de treinamento e validação reservada para uso 
no notebook **[02_ML](02_ML.ipynb)**

**Escopo:** os resultados descrevem os padrões observados na base e a capacidade de reproduzir os rótulos existentes. Não constituem validação de diagnóstico clínico.

---

<div style="background-color: #f5f0dc; padding: 12px; border-radius: 6px;">

> **Nota metodológica**  
> A base contém 3.772 registros e 30 colunas. A classe `P` representa 92,29% das observações e a classe `N`, 7,71%. Essa diferença caracteriza desbalanceamento e deve ser considerada tanto na divisão dos dados quanto na avaliação dos modelos. A codificação adotada na preparação será `P = 0` e `N = 1`.

</div>

### Principais achados

<div style="background-color: #DBEAFE; padding: 12px; border-radius: 6px;">

> **Desbalanceamento da variável-alvo**  
> A classe **P** corresponde a 92,29% dos registros (3.481), enquanto **N** representa 7,71% (291).

> **Pontos críticos de qualidade**  
> `TBG` apresenta ausência total; `TBG measured` é constante; `T3` possui 20,39% de valores ausentes; e foi identificado um registro com idade de 455 anos.

> **Diferenças entre as classes**  
> As medianas dos exames laboratoriais diferem entre `N` e `P`, especialmente em `TSH`, `TT4`, `T3` e `FTI`. Essas diferenças são exploratórias e serão investigadas na etapa de modelagem.

</div>

---

## Configuração do ambiente

A etapa configura os caminhos do projeto, importa as bibliotecas e carrega as funções auxiliares utilizadas nas análises e visualizações.

In [1]:
# # Configurações gerais

from setup_notebook import *
COR=color_palette21
COR_R=COR[::-1]

COR2=color_palette41
COR2_R=COR2[::-1]

sns.set_palette(color_palette21)
sns.set_style("whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

print(f"Projeto        : {PROJECT_ROOT}")
print(f"Notebook       : {NOTEBOOK_DIR}")
print("Bibliotecas e funções auxiliares carregadas.")
sns.color_palette(COR)

Projeto        : /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo
Notebook       : /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/notebook
Bibliotecas e funções auxiliares carregadas.


[(0.0, 0.2980392156862745, 0.2980392156862745),
 (0.0, 0.4, 0.4),
 (0.0, 0.5019607843137255, 0.5019607843137255),
 (0.09803921568627451, 0.5686274509803921, 0.5686274509803921),
 (0.1607843137254902, 0.6392156862745098, 0.6392156862745098),
 (0.25098039215686274, 0.7098039215686275, 0.7098039215686275),
 (0.3333333333333333, 0.7803921568627451, 0.7803921568627451),
 (0.4, 0.8509803921568627, 0.8509803921568627),
 (0.5019607843137255, 0.9254901960784314, 0.9254901960784314),
 (0.6, 1.0, 1.0),
 (1.0, 0.8352941176470589, 0.5019607843137255),
 (1.0, 0.7686274509803922, 0.3764705882352941),
 (1.0, 0.6980392156862745, 0.25098039215686274),
 (1.0, 0.6274509803921569, 0.12549019607843137),
 (1.0, 0.5568627450980392, 0.0),
 (1.0, 0.48627450980392156, 0.0),
 (1.0, 0.39215686274509803, 0.0),
 (1.0, 0.2980392156862745, 0.0),
 (1.0, 0.2, 0.0),
 (1.0, 0.10196078431372549, 0.0),
 (1.0, 0.0, 0.0)]

## 2. Carregamento e inspeção inicial

In [3]:
DATA_PATH = PROJECT_ROOT / "data" / "Hypothyroid.csv"

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Base não encontrada: {DATA_PATH}")

# Preservar a leitura original e criar cópia para análise
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.replace("?", np.nan).copy()

display(df.sample(min(5, len(df)), random_state=42))

# Resumo inicial da estrutura e da qualidade
inital_describe(df, simple=True)

,age,sex,on thyroxine,query on thyroxine,on antithyroid medication,sick,pregnant,thyroid surgery,I131 treatment,query hypothyroid,query hyperthyroid,lithium,goitre,tumor,hypopituitary,psych,TSH measured,TSH,T3 measured,T3,TT4 measured,TT4,T4U measured,T4U,FTI measured,FTI,TBG measured,TBG,referral source,binaryClass
270,26,F,f,f,f,f,f,f,f,f,f,f,f,f,f,f,t,2.1,t,2.2,t,95,t,1,t,95,f,NaN,other,P
2071,26,F,f,f,f,f,f,f,f,f,f,f,f,t,f,f,t,6.2,f,NaN,t,86,t,0.9,t,95,f,NaN,other,N
3660,52,M,t,f,f,f,f,f,f,f,f,f,f,f,f,f,t,0.2,t,2.3,t,116,t,0.94,t,123,f,NaN,other,P
3207,25,F,f,f,f,f,f,f,f,f,f,f,t,f,f,f,t,2,t,2.2,t,123,t,1.52,t,81,f,NaN,other,P
1675,29,F,f,f,f,f,f,f,f,f,f,f,f,f,f,f,t,0.84,t,1.5,t,100,t,0.74,t,135,f,NaN,other,P


📊 ANÁLISE EXPLORATÓRIA DO DATAFRAME

📈 DIMENSÕES DO DATASET:
   • 3772 linhas
   • 30 colunas
   • Total de células: 113160

🔧 TIPOS DE DADOS:
   • str: 30 colunas

🔍 VERIFICAÇÃO DE QUALIDADE DOS DADOS

📝 REGISTROS DUPLICADOS:
   • Total: 61
   • Percentual: 1.62%

❌ VALORES NULOS:
   • Total: 6064
   • Percentual: 5.36%

📊 COLUNAS COM VALORES NULOS:
   • age: 1 nulos (0.03%)
   • sex: 150 nulos (3.98%)
   • TSH: 369 nulos (9.78%)
   • T3: 769 nulos (20.39%)
   • TT4: 231 nulos (6.12%)
   • T4U: 387 nulos (10.26%)
   • FTI: 385 nulos (10.21%)
   • TBG: 3772 nulos (100.00%)

✅ ANÁLISE CONCLUÍDA


## 3. Identificação das variáveis

In [4]:
target = "binaryClass"

num_cols = ["age", "TSH", "T3", "TT4", "T4U", "FTI"]

binary_cols = [
    "on thyroxine",
    "query on thyroxine",
    "on antithyroid medication",
    "sick",
    "pregnant",
    "thyroid surgery",
    "I131 treatment",
    "query hypothyroid",
    "query hyperthyroid",
    "lithium",
    "goitre",
    "tumor",
    "hypopituitary",
    "psych",
    "TSH measured",
    "T3 measured",
    "TT4 measured",
    "T4U measured",
    "FTI measured",
    "TBG measured"]

cat_cols = ["sex", "referral source"]

colunas_esperadas = [target] + num_cols + binary_cols + cat_cols + ["TBG"]
colunas_ausentes = [col for col in colunas_esperadas if col not in df.columns]

if colunas_ausentes:
    raise ValueError(f"Colunas esperadas não encontradas: {colunas_ausentes}")

print(f"Variável-alvo: {target}")
print(f"Variáveis numéricas: {len(num_cols)}")
print(f"Variáveis binárias: {len(binary_cols)}")
print(f"Variáveis categóricas: {len(cat_cols)}")
print(f"Variável sem informação útil: TBG")
print(f"Total de colunas consideradas: {1 + len(num_cols) + len(binary_cols) + len(cat_cols) + 1}")

# display(
#     pd.DataFrame({
#         "grupo": (
#             ["Alvo"] +
#             ["Numérica"] * len(num_cols) +
#             ["Binária"] * len(binary_cols) +
#             ["Categórica"] * len(cat_cols) +
#             ["Sem informação"] 
#         ),
#         "variavel": [target] + num_cols + binary_cols + cat_cols + ["TBG"]
#     })
# )

Variável-alvo: binaryClass
Variáveis numéricas: 6
Variáveis binárias: 20
Variáveis categóricas: 2
Variável sem informação útil: TBG
Total de colunas consideradas: 30


## 4. Qualidade e consistência dos dados

### 4.1 Valores ausentes

In [5]:
# Percentual de valores ausentes por variável
missing_pct = (df.isna().mean().mul(100).loc[lambda x: x > 0].sort_values(ascending=True))
n_pct = len(missing_pct)

colors = COR[n_pct+1:0:-1] 

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.barh(missing_pct.index,missing_pct.values,
               color=colors,
               edgecolor="white",
               linewidth=0.6)
for i, value in enumerate(missing_pct.values):
    ax.text(value + 1, i, f"{value:.2f}%", va="center")

ax.set_title("Percentual de valores ausentes por variável",fontsize=13,fontweight="bold")
ax.set_xlabel("Valores ausentes (%)")
ax.set_ylabel("")
ax.set_xlim(0, 110)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()



Figura salva: /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/docs/figuras/eda_01.png


**Comentário**

A inspeção inicial da base identificou diferentes níveis de ausência de informação entre as variáveis. A situação mais crítica ocorre em `TBG`, que não apresenta valores preenchidos em nenhum dos 3.772 registros. Por esse motivo, essa variável não oferece informação para a análise e será excluída das etapas posteriores.

Entre as variáveis com ausência parcial, `T3` apresenta a maior proporção de valores ausentes (20,39%), seguida por `T4U` (10,26%), `FTI` (10,21%) e `TSH` (9,78%). Também foram identificadas ausências em `TT4` (6,12%), `sex` (3,98%) e, em menor proporção, `age`.

A presença de valores ausentes não implica, necessariamente, a exclusão dos registros. As variáveis com informação parcial serão preservadas nesta etapa exploratória, deixando a definição das estratégias de imputação para o pré-processamento dos dados de modelagem.

### 4.2 Registros duplicados

A contagem abaixo distingue duplicatas adicionais (`duplicated()`) do total de registros que pertencem a grupos duplicados (`keep=False`). Como o conjunto não apresenta identificador individual, registros idênticos não podem ser atribuídos com certeza à mesma pessoa.

In [6]:
n_total=len(df)
n_duplicatas_adicionais = int(df.duplicated().sum())
df_dup = df.loc[df.duplicated(keep=False)].copy()
n_registros_em_grupos_duplicados = len(df_dup)
n_sem_duplicacao = len(df)-len(df_dup)
n_excedentes = int(df.duplicated().sum())
n_representantes = df_dup.drop_duplicates().shape[0]

resumo_duplicatas = pd.DataFrame({"indicador": ["Registros totais",
                                                "Duplicatas adicionais",
                                                "Sem duplicacao",
                                                "Registros envolvidos em grupos duplicados",
                                                "Percentual de duplicatas adicionais",
                                                "Percentual de registros envolvidos"],
                                  "valor": [n_total,
                                            n_duplicatas_adicionais,
                                            n_sem_duplicacao,
                                            n_registros_em_grupos_duplicados,
                                            n_duplicatas_adicionais / len(df) * 100,
                                            n_registros_em_grupos_duplicados / len(df) * 100]})


# ---------- PRINT FORMATADO ----------
print("=" * 62)
print("RESUMO DE DUPLICATAS".center(62))
print("=" * 62)

for _, row in resumo_duplicatas.iterrows():
    indicador = row["indicador"]
    valor = row["valor"]

    # Percentuais com 2 casas + "%"; contagens como inteiro
    if "Percentual" in indicador:
        valor_fmt = f"{valor:>8.2f} %"
    else:
        valor_fmt = f"{int(valor):>8,d}".replace(",", ".")

    print(f"{indicador:<45} {valor_fmt}")

print("=" * 62)

# Alerta condicional (opcional)
if n_duplicatas_adicionais == 0:
    print("✅ Nenhuma duplicata encontrada.")
else:
    pct = n_duplicatas_adicionais / len(df) * 100
    print(f"⚠️  {n_duplicatas_adicionais} duplicatas adicionais "
          f"({pct:.2f}% do dataset).")


# Composição da base
categorias = ["Sem duplicação","Representantes dos grupos duplicados","Ocorrências excedentes"]

valores = [n_sem_duplicacao,n_representantes,n_excedentes]

fig, ax = plt.subplots(figsize=(9, 6))

wedges, texts, autotexts = ax.pie(valores,
    labels=None,
    autopct=lambda pct: f"{pct:.2f}%".replace(".", ",")
        if pct >= 5 else "",
    startangle=90,
    colors=cor_N(color_palette21,3),
    wedgeprops={"width": 0.45, "edgecolor": "white"},
    pctdistance=0.78,
    textprops={"fontsize": 10})

# Total no centro
ax.text(
    0, 0,
    f"{n_total:,}".replace(",", ".") + "\nRegistros",
    ha="center",
    va="center",
    fontsize=13,
    fontweight="bold"
)

# Legenda com quantidades e percentuais
legenda = [
    f"{cat}: {valor:,} ({valor / n_total * 100:.2f}%)".replace(",", ".").replace(".", ",", 1)
    for cat, valor in zip(categorias, valores)
]

ax.legend(
    wedges,
    legenda,
    title="Composição da base",
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=False
)

ax.set_title(
    "Participação dos registros duplicados na base",
    fontsize=13,
    fontweight="bold"
)

ax.set_aspect("equal")
plt.tight_layout()
plt.show()





                     RESUMO DE DUPLICATAS                     
Registros totais                                 3.772
Duplicatas adicionais                               61
Sem duplicacao                                   3.678
Registros envolvidos em grupos duplicados           94
Percentual de duplicatas adicionais               1.62 %
Percentual de registros envolvidos                2.49 %
⚠️  61 duplicatas adicionais (1.62% do dataset).
Figura salva: /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/docs/figuras/eda_02.png


#### **Comentários**

Dos 3.772 registros originais, 94 (2,49%) participam de grupos com registros integralmente idênticos. Esses registros correspondem a 33 padrões distintos, dos quais uma ocorrência de cada padrão será preservada, enquanto as 61 ocorrências excedentes serão removidas.

Um aspecto observado é que todos os 94 registros envolvidos pertencem à classe `P`. Essa característica é descritiva e não permite determinar a origem das repetições, tampouco afirmar que correspondam a uma mesma pessoa. Para a preparação dos dados, optou-se pela remoção das ocorrências excedentes, preservando uma representação de cada padrão identificado.

### 4.3 Verificação da variável idade

A idade é convertida temporariamente para formato numérico. Valores superiores a 100 anos são sinalizados para investigação; a decisão de tratamento será aplicada na cópia analítica, preservando `df_raw` e `df`.

In [7]:
age_temp = pd.to_numeric(df["age"], errors="coerce")

age_stats = age_temp.describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

#display(age_stats.to_frame("age"))

idade_invalida = age_temp > 100

# print(f"Registros com idade superior a 100: {int(idade_invalida.sum())}")
# print(f"Registros com idade ausente/não conversível: {int(age_temp.isna().sum())}")

# display(df.loc[idade_invalida | age_temp.isna(), [ "age", target ]])

#### **Comentários**

A distribuição das idades permite observar o perfil etário dos participantes e identificar possíveis valores discrepantes. A análise evidencia a presença de um registro com idade igual a 455 anos, valor incompatível com a faixa etária humana e que será tratado como uma inconsistência nos dados.

Esse registro não será excluído da base original. Para as etapas posteriores de modelagem, a idade será considerada ausente (`NaN`), preservando a observação e evitando que um valor inválido interfira nas análises estatísticas e no treinamento dos modelos.

### 4.4 Distribuição da variável-alvo

A base apresenta as classes `P` e `N`. Para a etapa de modelagem, será utilizada a codificação `P = 0` e `N = 1`, mantendo a interpretação da classe positiva conforme a documentação do conjunto.

In [8]:
target_dist = pd.DataFrame({
    "contagem": df[target].value_counts(dropna=False),
    "percentual": df[target].value_counts(normalize=True, dropna=False) * 100
})

display(target_dist.round(3))
out=bar_bar_cat(df, target, figsize=(8, 5))

,contagem,percentual
binaryClass,,
P,3481,92.285
N,291,7.715


Figura salva: /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/docs/figuras/eda_03.png


#### **Comentários**

A distribuição da variável-alvo evidencia um desequilíbrio entre as classes do conjunto de dados. A classe `P` representa 92,29% das observações (3.481 registros), enquanto a classe `N` corresponde a 7,71% (291 registros).

Essa diferença caracteriza um problema de desbalanceamento de classes, que deverá ser considerado durante a modelagem. A predominância da classe `P` pode favorecer modelos que priorizem sua classificação, tornando a acurácia isolada uma métrica insuficiente para avaliar o desempenho.

Dessa forma, serão utilizadas métricas como precisão, recall, F1-score e PR-AUC, permitindo avaliar de maneira mais adequada a capacidade dos modelos de identificar a classe minoritária.

Para a etapa de modelagem, será adotada a codificação `P = 0` e `N = 1`, preservando a correspondência com os rótulos originais do conjunto de dados.

## 5. Análise das variáveis numéricas

A análise considera seis variáveis numéricas: idade e cinco exames laboratoriais. As conversões e os tratamentos exploratórios são realizados em uma cópia da base, preservando os dados carregados originalmente.

In [9]:
df_eda = df.copy()

for col in num_cols:
    df_eda[col] = pd.to_numeric(df_eda[col], errors="coerce")

n_idades_invalidas = int((df_eda["age"] > 100).sum())
df_eda.loc[df_eda["age"] > 100, "age"] = np.nan

print(f"Idades inválidas convertidas em ausente: {n_idades_invalidas}")
print(f"Ausências em age após o tratamento: {int(df_eda['age'].isna().sum())}")

#display(df_eda[num_cols].describe().T.round(3))

# Distribuição das variáveis numéricas após o tratamento
fig, ax = plt.subplots(figsize=(9, 5))

sns.boxplot(
    data=df_eda[num_cols],
    orient="h",
    ax=ax
)

ax.set_title(
    "Distribuição das variáveis numéricas",
    fontsize=13,
    fontweight="bold"
)
ax.set_xlabel("Valores")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

Idades inválidas convertidas em ausente: 1
Ausências em age após o tratamento: 2
Figura salva: /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/docs/figuras/eda_04.png


#### **Comentários**

As variáveis numéricas foram convertidas para seus respectivos formatos, garantindo a consistência dos dados para as análises posteriores. Durante esse procedimento, foi identificada uma idade inválida (455 anos), convertida em valor ausente (`NaN`), sem exclusão do registro. Após o tratamento, a variável `age` passou a apresentar duas observações ausentes.

As estatísticas descritivas mostram que a idade dos participantes apresenta média de 51,63 anos e mediana de 54 anos. Entre as demais variáveis, destacam-se os valores máximos de `TSH` (530), `TT4` (430) e `FTI` (395), indicando a presença de valores extremos que merecem atenção no pré-processamento.

O boxplot permite observar a dispersão e os possíveis valores extremos das variáveis numéricas. Como as variáveis possuem escalas e unidades distintas, a comparação direta de suas amplitudes deve ser feita com cautela.

### 5.2 Distribuição das variáveis numéricas

Os histogramas apresentam a distribuição percentual de cada variável. A tabela retornada pela função auxiliar registra os intervalos, percentuais e contagens utilizados nos gráficos.

In [10]:
out_num = mult_plt2(
    df_eda[num_cols],
    kind="hist",
    ncols=3,
    max_bins=15,
    figsize=(16, 12),
    suptitle="Distribuição das variáveis numéricas"
)

#display(out_num)

Figura salva: /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/docs/figuras/eda_05.png


#### **Comentários**

A distribuição das variáveis numéricas revela comportamentos distintos entre idade e indicadores laboratoriais.

- **Idade (`age`):** apresenta maior concentração entre 36 e 76 anos, com destaque para as faixas em torno de 56 e 66 anos, que reúnem as maiores frequências.
- **TSH:** apresenta distribuição fortemente assimétrica à direita. Aproximadamente 98,35% dos registros estão concentrados no primeiro intervalo, enquanto uma pequena parcela apresenta valores elevados, chegando a aproximadamente 530.
- **T3:** apresenta maior concentração nos intervalos próximos de 1,55 e 2,55, que juntos representam aproximadamente 84,15% das observações.
- **TT4:** concentra aproximadamente 87,58% dos registros nos intervalos próximos de 77 e 127.
- **T4U:** apresenta maior concentração no intervalo próximo de 0,95 (47%), com predominância de valores entre os intervalos de 0,75 e 1,15.
- **FTI:** apresenta concentração expressiva nos intervalos próximos de 77 e 127, que juntos correspondem a aproximadamente 89,10% das observações.

As distribuições mostram diferentes níveis de concentração e assimetria. O `TSH` se destaca pela forte assimetria e pela presença de valores extremos, enquanto `T3`, `TT4`, `T4U` e `FTI` apresentam maior concentração em intervalos específicos. Os valores mencionados para os histogramas representam centros de intervalos, não necessariamente valores individuais observados.

### 5.3 Distribuição e valores extremos dos exames laboratoriais

Esta etapa aprofunda a análise dos cinco exames laboratoriais, reunindo estatísticas descritivas, proporção de ausências, assimetria e boxplots.

In [11]:
lab_cols = ["TSH", "T3", "TT4", "T4U", "FTI"]

stats_lab = df_eda[lab_cols].describe().T
stats_lab["missing"] = df_eda[lab_cols].isna().sum()
stats_lab["missing_pct"] = df_eda[lab_cols].isna().mean() * 100
stats_lab["skewness"] = df_eda[lab_cols].skew()

#display(stats_lab.round(3))

out1=mult_plt2(
    df_eda[lab_cols],
    kind="box",
    ncols=3,
    figsize=(16, 9),
    suptitle="Boxplots dos exames laboratoriais"
)

Figura salva: /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/docs/figuras/eda_06.png


#### **Comentários**

A análise estatística e gráfica dos exames laboratoriais evidencia diferenças na dispersão, assimetria e presença de valores extremos entre as variáveis.

O **TSH** apresenta a maior assimetria positiva (13,883), com média (5,087) bastante superior à mediana (1,400) e valor máximo de 530, indicando forte concentração de observações em valores menores e uma cauda prolongada à direita.

O **T3** também apresenta assimetria positiva (1,731), além da maior proporção de valores ausentes entre os exames (20,39%). `TT4`, `T4U` e `FTI` apresentam assimetrias positivas mais moderadas, com valores máximos consideravelmente superiores aos respectivos terceiros quartis.

Quanto aos valores ausentes, destacam-se `T3` (20,39%), `T4U` (10,26%), `FTI` (10,21%) e `TSH` (9,78%). Essas diferenças devem ser consideradas na preparação dos dados.

Em conjunto, os resultados indicam distribuições assimétricas e possíveis valores extremos. Como não há elementos nesta análise para determinar se esses valores são erros ou variações legítimas, eles serão mantidos inicialmente e avaliados durante o pré-processamento.

### 5.4 Estatísticas descritivas por classe

As estatísticas são agrupadas pela variável-alvo para comparar medidas de posição e dispersão das variáveis numéricas entre as classes `N` e `P`.

In [12]:
stats_class = (
    df_eda
    .groupby(target)[num_cols]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(3)
)

display(stats_class)

# Comparação das distribuições por classe
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

palette_target = {
    "N": color_palette21[3],
    "P": color_palette21[16]
}

for ax, col in zip(axes, num_cols):
    sns.boxplot(
        data=df_eda,
        x=target,
        y=col,
        hue=target,
        order=["N", "P"],
        hue_order=["N", "P"],
        palette=palette_target,
        legend=False,
        ax=ax
    )
    ax.set_title(col, fontsize=12, fontweight="bold")
    ax.set_xlabel("Classe")
    ax.set_ylabel("Valor")
    ax.grid(axis="y", alpha=0.25)

for ax in axes[len(num_cols):]:
    ax.set_visible(False)

fig.suptitle(
    "Distribuição das variáveis numéricas por classe",
    fontsize=14,
    fontweight="bold"
)
plt.tight_layout()
plt.show()

age                                     TSH                \
            count   mean median    std   min    max count   mean median   
binaryClass                                                               
N             291 51.515 55.000 19.109 1.000 88.000   291 39.234 12.000   
P            3479 51.638 54.000 18.975 1.000 94.000  3112  1.894  1.200   

                                    T3                                   TT4  \
               std   min     max count  mean median   std   min    max count   
binaryClass                                                                    
N           74.384 0.015 530.000   245 1.475  1.500 0.761 0.200  4.100   286   
P            4.744 0.005 145.000  2758 2.061  2.000 0.816 0.050 10.600  3255   

                                                    T4U                     \
               mean  median    std    min     max count  mean median   std   
binaryClass                                                                  
N            72.915  76.500 34.913  2.000 148.000   266 1.015  1.010 0.185   
P           111.430 105.000 33.948 19.000 430.000  3119 0.993  0.970 0.196   

                          FTI                                        
              min   max count    mean  median    std    min     max  
binaryClass                                                          
N           0.560 1.650   266  73.247  77.500 33.452  2.000 153.000  
P           0.250 2.320  3121 113.642 108.000 31.065 17.000 395.000

Figura salva: /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/docs/figuras/eda_07.png


#### **Comentários**

A comparação das estatísticas descritivas entre as classes revela diferenças importantes principalmente nos indicadores laboratoriais, enquanto a variável idade apresenta distribuições bastante semelhantes.

A classe `N` apresenta valores médios e medianos de `TSH` substancialmente superiores aos da classe `P`. Também são observadas diferenças em `TT4` e `FTI`, cujas médias e medianas são menores na classe `N`. Para `T3` e `T4U`, as diferenças entre as medidas centrais são menos expressivas.

A idade apresenta médias próximas entre as classes (`51,515` anos em `N` e `51,638` anos em `P`), indicando pouca diferença entre os grupos nessa variável.

Os boxplots permitem comparar a dispersão e os valores extremos em cada classe. Essas diferenças sugerem que os indicadores laboratoriais podem contribuir para distinguir os grupos, aspecto que será investigado na modelagem. Entretanto, as estatísticas descritivas, isoladamente, não permitem avaliar a capacidade preditiva das variáveis.

### 5.5 Comparação das medianas e dos quartis

A diferença entre medianas é calculada como `N - P`. A variação percentual utiliza a mediana de `P` como referência. Os intervalos interquartis (Q1–Q3) complementam a comparação ao mostrar a amplitude dos 50% centrais das observações.

In [13]:
comparacao_medianas = (df_eda.groupby(target)[num_cols].median().T)

comparacao_medianas["diferenca_N_P"] = (comparacao_medianas["N"] - comparacao_medianas["P"])

comparacao_medianas["variacao_percentual"] = (comparacao_medianas["diferenca_N_P"]/ comparacao_medianas["P"].replace(0, np.nan)) * 100

#display(comparacao_medianas.round(2))

quartis = (df_eda.groupby(target)[num_cols].quantile([0.25, 0.50, 0.75]).unstack(level=0).round(3))

#display(quartis)


# 

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):

    for classe, cor, deslocamento in [
        ("N", color_palette21[3], -0.12),
        ("P", color_palette21[16], 0.12)
    ]:
        q1 = quartis.loc[0.250, (col, classe)]
        mediana = quartis.loc[0.500, (col, classe)]
        q3 = quartis.loc[0.750, (col, classe)]

        ax.plot([q1, q3],[deslocamento, deslocamento],
                color=cor,
                linewidth=5,solid_capstyle="round")
        ax.scatter(mediana,deslocamento,
                   color=cor,s=90,zorder=3,label=classe)
        ax.set_title(col, fontsize=12, fontweight="bold")
        ax.set_yticks([])
        ax.set_xlabel("Valor")
        ax.grid(axis="x", alpha=0.25)

axes[-1].set_visible(False)

fig.suptitle("Medianas e intervalos interquartis por classe",fontsize=14,fontweight="bold")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Classe", loc="upper right")

plt.tight_layout()
plt.show()

Figura salva: /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/docs/figuras/eda_08.png


#### **Comentários**


A comparação das medianas evidencia diferenças entre as classes principalmente nos indicadores laboratoriais. O `TSH` apresenta mediana de 12,0 na classe `N`, contra 1,2 na classe `P`, correspondendo a uma diferença de 900%. Em sentido oposto, `T3`, `TT4` e `FTI` apresentam medianas menores na classe `N`, com diferenças de -25,00%, -27,14% e -28,24%, respectivamente.

As diferenças em `age` e `T4U` são menos expressivas: 1 ano para idade e 0,04 para `T4U`. Esse resultado reforça que as diferenças nas medidas centrais são mais evidentes nos exames laboratoriais do que na idade.

Os quartis complementam essa comparação ao mostrar a dispersão dos valores em cada grupo. Destaca-se o `TSH` na classe `N`, cujo intervalo interquartil vai de 8,4 a 34,5, enquanto na classe `P` varia de 0,4 a 2,2. Para `TT4` e `FTI`, os intervalos interquartis da classe `N` também se encontram em níveis inferiores aos da classe `P`.

Em conjunto, os resultados mostram diferenças nas distribuições dos exames laboratoriais entre as classes. Essas evidências são exploratórias e não demonstram, isoladamente, associação causal ou capacidade de classificação.

### 5.6 Correlação entre variáveis numéricas

A matriz de correlação de Spearman é utilizada para investigar associações monotônicas entre idade e exames laboratoriais. Essa medida foi escolhida por ser menos sensível à assimetria e aos valores extremos observados nas distribuições. A análise é exploratória: correlação não implica causalidade nem, isoladamente, determina a importância preditiva das variáveis.

In [14]:
# Correlação de Spearman entre as variáveis numéricas
corr_spearman = df_eda[num_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_spearman,
    annot=True,
    fmt=".2f",
    cmap=COR,
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlação de Spearman"},
    ax=ax
)
ax.set_title("Correlação de Spearman entre variáveis numéricas", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

display(corr_spearman.round(3))

Figura salva: /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/docs/figuras/eda_09.png


,age,TSH,T3,TT4,T4U,FTI
age,1.000,-0.014,-0.290,-0.023,-0.150,0.081
TSH,-0.014,1.000,-0.231,-0.350,0.069,-0.412
T3,-0.290,-0.231,1.000,0.418,0.447,0.110
TT4,-0.023,-0.350,0.418,1.000,0.430,0.742
T4U,-0.150,0.069,0.447,0.430,1.000,-0.193
FTI,0.081,-0.412,0.110,0.742,-0.193,1.000


#### **Comentários**

A análise de correlação de Spearman identificou associações relevantes entre algumas variáveis laboratoriais. A maior correlação positiva ocorreu entre TT4 e FTI (0,742), seguida pelas relações entre T3 e T4U (0,447), TT4 e T4U (0,430) e T3 e TT4 (0,418).

Também foi observada correlação negativa moderada entre TSH e FTI (-0,412), enquanto a relação entre idade e T3 apresentou associação negativa mais discreta (-0,290). As demais combinações apresentaram correlações fracas.

Os resultados indicam que algumas medidas laboratoriais compartilham informações, especialmente TT4 e FTI. Essa sobreposição deve ser considerada na interpretação da importância das variáveis e do comportamento dos modelos. Ressalta-se que correlação não implica causalidade e que a matriz não permite, isoladamente, estabelecer relações clínicas.

### 5.7 Assimetria e dados ausentes

A seção reúne, para cada variável numérica, o percentual de valores ausentes, o coeficiente de assimetria e a mediana. O gráfico combina os dois primeiros indicadores para facilitar a identificação das variáveis que concentram ausências e das distribuições mais assimétricas.

In [15]:
resumo_numerico = pd.DataFrame({
    "tipo": "Numérica",
    "ausentes": df_eda[num_cols].isna().sum(),
    "ausentes_%": df_eda[num_cols].isna().mean() * 100,
    "assimetria": df_eda[num_cols].skew(),
    "mediana": df_eda[num_cols].median()
})

display(resumo_numerico.round(3))

fig, ax1 = plt.subplots(figsize=(10, 5))

dados = resumo_numerico.sort_values("ausentes_%", ascending=True)

y = np.arange(len(dados))

# Percentual de valores ausentes
ax1.barh(
    y,
    dados["ausentes_%"],
    alpha=0.75,
    label="Valores ausentes (%)"
)

ax1.set_yticks(y)
ax1.set_yticklabels(dados.index)
ax1.set_xlabel("Valores ausentes (%)")
ax1.set_ylabel("Variáveis")
ax1.set_xlim(0, dados["ausentes_%"].max() * 1.25)
ax1.grid(axis="x", alpha=0.25)

# Assimetria em eixo secundário
ax2 = ax1.twiny()

ax2.scatter(
    dados["assimetria"],
    y,
    color="darkorange",
    s=85,
    label="Assimetria"
)

ax2.set_xlabel("Coeficiente de assimetria")
ax2.axvline(0, color="gray", linestyle="--", alpha=0.7)

ax1.set_title(
    "Assimetria e percentual de valores ausentes",
    fontsize=13,
    fontweight="bold"
)

# Legenda conjunta
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="lower right")

plt.tight_layout()
plt.show()

,tipo,ausentes,ausentes_%,assimetria,mediana
age,Numérica,2,0.053,-0.211,54.000
TSH,Numérica,369,9.783,13.883,1.400
T3,Numérica,769,20.387,1.731,2.000
TT4,Numérica,231,6.124,1.268,103.000
T4U,Numérica,387,10.260,1.233,0.980
FTI,Numérica,385,10.207,1.345,107.000


Figura salva: /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/docs/figuras/eda_10.png


#### **Comentários**

A análise conjunta da assimetria e dos valores ausentes evidencia características distintas entre as variáveis numéricas.

O `TSH` apresenta a maior assimetria positiva (13,883), indicando uma distribuição fortemente concentrada em valores baixos, com uma cauda alongada à direita. As demais variáveis laboratoriais também apresentam assimetria positiva, embora menos acentuada, com coeficientes entre 1,233 e 1,731. A idade apresenta assimetria ligeiramente negativa (-0,211).

Em relação aos valores ausentes, `T3` apresenta a maior proporção (20,39%), seguida por `T4U` (10,26%), `FTI` (10,21%) e `TSH` (9,78%). A variável `age` apresenta duas observações ausentes, correspondentes a aproximadamente 0,05% da base.

Essas características deverão ser consideradas no pré-processamento, especialmente na definição das estratégias de imputação e transformação das variáveis. O gráfico utiliza eixos com escalas distintas para os dois indicadores; portanto, não deve ser interpretado como evidência de relação estatística entre assimetria e ausência de dados.

## 6. Análise das variáveis categóricas

As variáveis binárias e categóricas são examinadas quanto à composição das classes, à frequência das categorias, à presença de ausências e à concentração em categorias dominantes.

### 6.1 Distribuição das categorias por classe

In [16]:
cat_analysis = binary_cols + cat_cols
results_cat = []

for col in cat_analysis:
    temp = df[[col, target]].copy()
    temp[col] = temp[col].fillna("Ausente")

    counts = pd.crosstab(temp[col], temp[target], dropna=False)
    percentages = pd.crosstab(
        temp[col],
        temp[target],
        normalize="index",
        dropna=False
    ) * 100

    for category in counts.index:
        results_cat.append({
            "variavel": col,
            "categoria": category,
            "n": int(counts.loc[category].sum()),
            "N_contagem": int(counts.loc[category].get("N", 0)),
            "N_percentual": percentages.loc[category].get("N", 0),
            "P_contagem": int(counts.loc[category].get("P", 0)),
            "P_percentual": percentages.loc[category].get("P", 0)
        })

df_cat_analysis = pd.DataFrame(results_cat)

#display(df_cat_analysis.sort_values(["variavel", "categoria"]).round(2).reset_index(drop=True))

n_variaveis_cat = df_cat_analysis["variavel"].nunique()
print(f"Categorias avaliadas: {n_variaveis_cat} variáveis; {len(df_cat_analysis)} combinações variável-categoria.")

Categorias avaliadas: 22 variáveis; 47 combinações variável-categoria.


### 6.2 Visualização das variáveis clínicas e categóricas

Os gráficos mostram a composição percentual das classes dentro de cada categoria. A seleção inclui características clínicas e as variáveis `sex` e `referral source`. A categoria `Ausente` é mantida quando aplicável.

In [17]:

clinical_binary = [
    "on thyroxine",
    "on antithyroid medication",
    "sick",
    "pregnant",
    "thyroid surgery",
    "I131 treatment",
    "query hypothyroid",
    "query hyperthyroid",
    "lithium",
    "goitre",
    "tumor",
    "hypopituitary",
    "psych"
]

cat_plot_cols = clinical_binary + ["sex", "referral source"]


palette_target = {
    "N": color_palette21[3],
    "P": color_palette21[15]
}



fig, axes = plt.subplots(5, 3, figsize=(17, 22))

for ax, col in zip(axes.flat, cat_plot_cols):
    temp = df[[col, target]].copy()
    temp[col] = temp[col].fillna("Ausente")

    tab = pd.crosstab(
        temp[col],
        temp[target],
        normalize="index",
        dropna=False
    ) * 100

    tab = tab.reindex(columns=["N", "P"], fill_value=0)

    tab.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=[palette_target["N"], palette_target["P"]]
    )

    ax.set_title(col)
    ax.set_ylabel("Percentual (%)")
    ax.set_xlabel("")
    ax.legend(title="Classe")
    ax.tick_params(axis="x", rotation=0)

for ax in axes.flat[len(cat_plot_cols):]:
    ax.set_visible(False)

plt.suptitle(
    "Distribuição das classes por características clínicas",
    fontsize=16,
    y=1.01
)
plt.tight_layout()
plt.show()

Figura salva: /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/docs/figuras/eda_11.png


#### **Comentários**


A análise mostra que a classe `P` é predominante em praticamente todas as categorias, mantendo proporções próximas à distribuição geral da base (92,29% para `P` e 7,71% para `N`). Entretanto, algumas categorias apresentam diferenças relevantes nessa composição.

Os principais destaques são:

- **`query hypothyroid`:** entre os registros com essa condição marcada (`t`), a classe `N` representa 16,67%, proporção superior à observada na distribuição geral.
- **`TSH measured`:** todos os 369 registros sem medição de TSH (`f`) pertencem à classe `P`. Entre aqueles com medição (`t`), a classe `N` representa 8,55%.
- **`TT4 measured`:** entre os registros sem medição (`f`), 97,84% pertencem à classe `P`, enquanto a classe `N` representa 2,16%.
- **`on thyroxine`:** na categoria `t`, a classe `P` corresponde a 98,06% dos registros.
- **`referral source`:** há variações entre as categorias. A origem `SVI` apresenta 9,38% de registros da classe `N`, enquanto `SVHC` apresenta 2,85%.

Algumas categorias possuem poucas observações, como `hypopituitary = t` (1 registro), `lithium = t` (18 registros) e `goitre = t` (34 registros). Suas proporções devem, portanto, ser interpretadas com cautela.

Embora a classe `P` predomine em todas as categorias apresentadas, algumas variáveis apresentam diferenças na composição entre as classes. Essas diferenças são relevantes para a exploração, mas não demonstram, isoladamente, associação estatística ou capacidade preditiva.

### 6.3 Qualidade e distribuição das variáveis categóricas

A avaliação reúne o percentual de valores ausentes, a quantidade de categorias distintas, a categoria mais frequente e sua participação entre os registros não ausentes. A tabela de frequências complementa o resumo ao apresentar a contagem e a proporção de cada categoria.

In [18]:
# Resumo da qualidade das variáveis categóricas
resumo_categorico = []
categorias = []

for col in cat_analysis:

    frequencias = df[col].fillna("Ausente").value_counts(dropna=False)
    frequencias_validas = df[col].value_counts(dropna=True)
    moda = df[col].mode(dropna=True)

    resumo_categorico.append({
        "variavel": col,
        "ausentes": int(df[col].isna().sum()),
        "ausentes_%": df[col].isna().mean() * 100,
        "categorias": int(df[col].nunique(dropna=True)),
        "categoria_mais_frequente": moda.iloc[0] if not moda.empty else np.nan,
        "frequencia_modal": int(frequencias_validas.iloc[0]) if not frequencias_validas.empty else 0
    })

    for categoria, quantidade in frequencias.items():
        categorias.append({
            "variavel": col,
            "categoria": categoria,
            "frequencia": int(quantidade),
            "percentual": quantidade / len(df) * 100
        })


df_resumo_categorico = pd.DataFrame(resumo_categorico)

df_frequencias = pd.DataFrame(categorias)

# Tabela de qualidade
#display(df_resumo_categorico.sort_values(["ausentes_%", "variavel"], ascending=[False, True]).round(2))
# # Frequência absoluta e percentual das categorias
#display(df_frequencias.sort_values(["variavel", "frequencia"], ascending=[True, False]).reset_index(drop=True).round(2))

# Percentual da categoria mais frequente em cada variável
df_plot = df_resumo_categorico.copy()

df_plot["modal_%"] = (
    df_plot["frequencia_modal"]
    / (len(df) - df_plot["ausentes"])
) * 100

df_plot = df_plot.sort_values("modal_%", ascending=True)



# Gráfico
fig, ax = plt.subplots(figsize=(10, 8))

ax.barh(
    df_plot["variavel"],
    df_plot["modal_%"],
    color=cor_N(COR2_R,22,0)
)

ax.set_title(
    "Concentração da categoria mais frequente por variável",
    fontsize=13,
    fontweight="bold"
)
ax.set_xlabel("Frequência da categoria mais frequente (%)")
ax.set_ylabel("")
ax.set_xlim(0, 105)
ax.grid(axis="x", alpha=0.25)

for i, value in enumerate(df_plot["modal_%"]):
    ax.text(value + 1, i, f"{value:.1f}%", va="center", fontsize=8)

plt.tight_layout()
plt.show()

Figura salva: /home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/docs/figuras/eda_12.png


#### **Comentários**

A análise das variáveis categóricas evidencia diferentes níveis de concentração das categorias, além de limitações na disponibilidade e diversidade das informações.

A variável `sex` é a única que apresenta valores ausentes, totalizando 150 registros (3,98%). Quanto à cardinalidade, a maioria das variáveis possui duas categorias, enquanto `referral source` apresenta cinco. Destaca-se `TBG measured`, que possui apenas uma categoria (`f`) em todos os registros, indicando ausência de variação.

O gráfico evidencia a concentração da categoria mais frequente em cada variável. `hypopituitary`, `goitre`, `I131 treatment` e `TBG measured` apresentam forte predominância de uma única categoria. Esse comportamento também é observado em `pregnant`, `thyroid surgery` e `sick`.

A tabela de frequências revela categorias com baixa representatividade, como `hypopituitary = t` (1 registro), `lithium = t` (18 registros) e `goitre = t` (34 registros). Esses casos devem ser interpretados com cautela. Variáveis constantes não oferecem informação discriminativa, enquanto categorias raras podem exigir atenção no pré-processamento. A variável `sex` também deverá receber tratamento para seus valores ausentes.

## 7. Síntese da qualidade dos dados

A auditoria consolida os principais problemas identificados nas etapas anteriores e registra as decisões previstas para a preparação da base destinada à modelagem.

In [19]:
n_tbg_ausentes = int(df["TBG"].isna().sum()) if "TBG" in df.columns else 0
n_tbg_medidos_unicos = (
    int(df["TBG measured"].nunique(dropna=False))
    if "TBG measured" in df.columns else np.nan
)
valor_tbg_measured = (
    df["TBG measured"].dropna().iloc[0]
    if "TBG measured" in df.columns and df["TBG measured"].notna().any()
    else np.nan
)

auditoria = pd.DataFrame([
    [
        "Valores ausentes",
        "TBG",
        f"{n_tbg_ausentes} ({df['TBG'].isna().mean() * 100:.2f}%)" if "TBG" in df.columns else "Coluna ausente",
        "Excluir da modelagem: sem observações disponíveis"
    ],
    [
        "Variável constante",
        "TBG measured",
        f"{n_tbg_medidos_unicos} valor(es) distinto(s); valor observado: {valor_tbg_measured}",
        "Excluir da modelagem: sem variabilidade"
    ],
    [
        "Valores ausentes",
        "T3",
        f"{df['T3'].isna().sum()} ({df['T3'].isna().mean() * 100:.2f}%)",
        "Tratar no pipeline de pré-processamento"
    ],
    [
        "Valores ausentes",
        "sex",
        f"{df['sex'].isna().sum()} ({df['sex'].isna().mean() * 100:.2f}%)",
        "Tratar no pipeline de pré-processamento"
    ],
    [
        "Inconsistência",
        "age",
        f"{int((age_temp > 100).sum())} valor(es) acima de 100",
        "Converter em ausente; não excluir o registro"
    ],
    [
        "Duplicatas exatas",
        "Dataset",
        f"{n_duplicatas_adicionais} duplicatas adicionais ({n_duplicatas_adicionais / len(df) * 100:.2f}%)",
        "Remover antes da divisão treino/validação"
    ],
    [
        "Assimetria",
        "TSH",
        f"Assimetria = {df_eda['TSH'].skew():.3f}",
        "Avaliar comportamento no processo de modelagem"
    ],
    [
        "Desbalanceamento",
        target,
        "; ".join(
            f"{classe}: {perc:.2f}%"
            for classe, perc in (df[target].value_counts(normalize=True) * 100).items()
        ),
        "Usar divisão estratificada e métricas adequadas"
    ]
], columns=["problema", "variavel", "evidencia", "decisao"])

display(auditoria)

,problema,variavel,evidencia,decisao
0,Valores ausentes,TBG,3772 (100.00%),Excluir da modelagem: sem observações disponíveis
1,Variável constante,TBG measured,1 valor(es) distinto(s); valor observado: f,Excluir da modelagem: sem variabilidade
2,Valores ausentes,T3,769 (20.39%),Tratar no pipeline de pré-processamento
3,Valores ausentes,sex,150 (3.98%),Tratar no pipeline de pré-processamento
4,Inconsistência,age,1 valor(es) acima de 100,Converter em ausente; não excluir o registro
5,Duplicatas exatas,Dataset,61 duplicatas adicionais (1.62%),Remover antes da divisão treino/validação
6,Assimetria,TSH,Assimetria = 13.883,Avaliar comportamento no processo de modelagem
7,Desbalanceamento,binaryClass,P: 92.29%; N: 7.71%,Usar divisão estratificada e métricas adequadas


---
## **Decisões para a preparação da base**

As decisões abaixo sintetizam o tratamento definido a partir das evidências observadas na análise exploratória.

## 8. Preparação da base analítica e divisão dos conjuntos

A preparação estabelece a base que será utilizada no notebook `02_ML`. As operações são realizadas na sequência documentada abaixo:

1. converter variáveis numéricas;
2. tratar a idade inválida;
3. remover `TBG` e `TBG measured`;
4. codificar o alvo (`P = 0`, `N = 1`);
5. remover duplicatas exatas;
6. dividir estratificadamente em 80% para treinamento e 20% para validação reservada.

A validação reservada deverá permanecer separada durante o desenvolvimento e a seleção dos modelos.

In [20]:
from sklearn.model_selection import train_test_split

# 1. Criar a base analítica a partir dos dados originais tratados
df_model = df.copy()

# 2. Converter variáveis numéricas
for col in num_cols:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

# 3. Tratar idade inválida sem excluir o registro
df_model.loc[df_model["age"] > 100, "age"] = np.nan

# 4. Remover variáveis sem informação útil para modelagem
colunas_excluir = [
    col for col in ["TBG", "TBG measured"]
    if col in df_model.columns
]
df_model = df_model.drop(columns=colunas_excluir)

# 5. Codificar a variável-alvo
mapa_target = {"P": 0, "N": 1}
df_model[target] = df_model[target].map(mapa_target)

if df_model[target].isna().any():
    raise ValueError("A variável-alvo contém valores ausentes ou rótulos não mapeados.")

# 6. Remover duplicatas exatas
n_antes_deduplicacao = len(df_model)
df_model = df_model.drop_duplicates().reset_index(drop=True)
n_removidos = n_antes_deduplicacao - len(df_model)

# 7. Separar preditores e alvo
X = df_model.drop(columns=[target])
y = df_model[target].astype(int)

# 8. Divisão estratificada 80/20
X_treino, X_validacao, y_treino, y_validacao = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 9. Reconstruir os conjuntos completos
df_treino = X_treino.copy()
df_treino[target] = y_treino

df_validacao = X_validacao.copy()
df_validacao[target] = y_validacao

# 10. Auditoria da preparação
print(f"Registros antes da deduplicação: {n_antes_deduplicacao:,}")
print(f"Duplicatas exatas removidas: {n_removidos:,}")
print(f"Registros após a deduplicação: {len(df_model):,}")
print(f"Variáveis excluídas: {colunas_excluir}")

print(f"\nBase analítica completa: {df_model.shape}")
print(f"Treinamento: {df_treino.shape}")
print(f"Validação reservada: {df_validacao.shape}")

distribuicao = pd.DataFrame({
    "Base completa": y.value_counts(normalize=True) * 100,
    "Treinamento": y_treino.value_counts(normalize=True) * 100,
    "Validação reservada": y_validacao.value_counts(normalize=True) * 100
}).rename(index={
    0: "P — classe 0",
    1: "N — classe 1"
})

display(distribuicao.round(2))

print("\nDuplicatas no conjunto de treinamento:", df_treino.duplicated().sum())
print("Duplicatas no conjunto de validação:", df_validacao.duplicated().sum())

assert set(df_treino.columns) == set(df_validacao.columns)
assert df_treino[target].notna().all()
assert df_validacao[target].notna().all()
assert df_treino.duplicated().sum() == 0
assert df_validacao.duplicated().sum() == 0

Registros antes da deduplicação: 3,772
Duplicatas exatas removidas: 61
Registros após a deduplicação: 3,711
Variáveis excluídas: ['TBG', 'TBG measured']

Base analítica completa: (3711, 28)
Treinamento: (2968, 28)
Validação reservada: (743, 28)


,Base completa,Treinamento,Validação reservada
binaryClass,,,
P — classe 0,92.160,92.150,92.190
N — classe 1,7.840,7.850,7.810



Duplicatas no conjunto de treinamento: 0
Duplicatas no conjunto de validação: 0


## 9. Exportação dos conjuntos para o ML02

Os conjuntos de treinamento e validação reservada são exportados em arquivos CSV no diretório `data` do projeto, para consumo pelo notebook de modelagem.

In [21]:
PATH_TREINO = PROJECT_ROOT / "data" / "df_treino.csv"
PATH_VALIDACAO = PROJECT_ROOT / "data" / "df_validacao.csv"

df_treino.to_csv(PATH_TREINO, index=False)
df_validacao.to_csv(PATH_VALIDACAO, index=False)

print("Arquivos exportados:")
print(PATH_TREINO)
print(PATH_VALIDACAO)

# Conferência dos arquivos gerados
print("\nTamanhos dos arquivos:")
print(f"Treinamento: {PATH_TREINO.stat().st_size / (1024 ** 2):.2f} MB")
print(f"Validação: {PATH_VALIDACAO.stat().st_size / (1024 ** 2):.2f} MB")

Arquivos exportados:
/home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/data/df_treino.csv
/home/akel/PycharmProjects/EBAC/Diagnostico_hipotireoidismo/data/df_validacao.csv

Tamanhos dos arquivos:
Treinamento: 0.21 MB
Validação: 0.05 MB


## 10. Conclusão da análise exploratória

A análise exploratória partiu de 3.772 registros e 30 colunas. A variável-alvo apresenta predominância da classe `P` (92,29%; 3.481 registros) sobre a classe `N` (7,71%; 291 registros), caracterizando desbalanceamento que deverá ser considerado na modelagem.

**Qualidade e consistência dos dados**

Foram identificados 61 registros excedentes em grupos de duplicatas exatas, correspondentes a 33 padrões repetidos. Também foi identificada uma idade inválida (455 anos), convertida em valor ausente na cópia analítica, sem exclusão do registro. A variável `TBG` está integralmente ausente e `TBG measured` apresenta apenas uma categoria, sem variabilidade. Entre as variáveis com ausência parcial, `T3` apresenta a maior proporção de ausências (20,39%).

**Comportamento das variáveis numéricas e correlação**

Os exames laboratoriais apresentam assimetria positiva, especialmente `TSH` (13,883), cuja média (5,087) é superior à mediana (1,400). A comparação entre classes mostra diferenças nas medidas centrais, com destaque para `TSH`, cuja mediana é 12,0 em `N` e 1,2 em `P`. A matriz de Spearman complementa a exploração ao examinar associações monotônicas entre as variáveis numéricas; seus coeficientes devem ser interpretados como relações entre variáveis, não como evidência causal. `TT4`, `FTI` e `T3` apresentam medianas menores na classe `N`. A idade, por sua vez, apresenta médias e medianas próximas entre os grupos.

**Comportamento das variáveis categóricas**

A variável `sex` apresenta 150 valores ausentes (3,98%). A maioria das variáveis binárias possui categorias fortemente desbalanceadas, e algumas categorias têm poucas observações. `TBG measured` é constante, enquanto `referral source` apresenta cinco categorias. As diferenças observadas na composição das classes são descritivas e não demonstram, isoladamente, associação estatística ou capacidade preditiva.

**Preparação e continuidade**

Após o tratamento da idade inválida, exclusão de `TBG` e `TBG measured` e remoção das 61 duplicatas excedentes, a base analítica contém 3.711 registros e 28 colunas. A divisão estratificada resultou em 2.968 registros para treinamento e 743 para validação reservada, mantendo proporções semelhantes entre as classes. Os arquivos `df_treino.csv` e `df_validacao.csv` são destinados ao notebook **[02_ML](notebook/02_ML.ipynb)**.

<div style="background-color: #E2E8F0; padding: 12px; border-radius: 6px;">

**Síntese final**

A base apresenta desbalanceamento de classes, ausências pontuais, uma variável integralmente ausente, uma variável constante, duplicatas exatas e diferenças nas distribuições dos exames laboratoriais entre as classes. As decisões de preparação foram documentadas e aplicadas antes da divisão estratificada dos dados. A etapa seguinte deverá avaliar os modelos com métricas adequadas ao desbalanceamento, mantendo o conjunto de validação reservado para a avaliação final.

</div>
